Импортируем библиотеки, собираем все данные

In [16]:
import os
import re
import pandas as pd

BASE_DIR = '/Users/sergey/PycharmProjects/SemAnalyzer/Keywords/Задание '
YEARS = [str(y) for y in range(2014, 2025)]

pattern = re.compile(r'^(.*?)_IMS_(\d{4})_rus\.txt$')

data = []

for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if 'Abstract' in file or 'KW' in file:
            continue
        match = pattern.match(file)
        if match and match.group(2) in YEARS:
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
            name = file  # имя файла целиком
            year = match.group(2)
            data.append({'name': name, 'text': text, 'year': year})

df = pd.DataFrame(data, columns=['name', 'text', 'year'])

df

,name,text,year
0,Kuzmina_IMS_2014_rus.txt,Социально-гуманитарная образовательная среда\n...,2014
1,Bogoraz_IMS_2014_rus.txt,Централизованные и распределенные социальные с...,2014
2,Grigoryeva_IMS_2014_rus.txt,Обучение пожилых ИКТ и возможности их\nтрудоус...,2014
3,Dyakova_IMS_2014_rus.txt,Использование информационных технологий\nдля б...,2014
4,Lizunova_IMS_2014_rus.txt,Безграничность медиапространства:\nбудущее или...,2014
...,...,...,...
137,Korablinov_IMS_2020_rus.txt,Подготовка набора данных для вопросно-ответног...,2020
138,Khokhlova_IMS_2020_rus.txt,Методы машинного обучения применительно к зада...,2020
139,Kuznetsova_IMS_2020_rus.txt,О возможности использования корпуса NOW в курс...,2020
140,Sokolova_IMS_2020_rus.txt,К вопросу о формировании набора отношений для ...,2020


SUMY

In [18]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

/Users/sergey/PycharmProjects/SemAnalyzer/.venvNew/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
[nltk_data] Downloading package punkt to /Users/sergey/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sergey/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [19]:
summarizer = LexRankSummarizer()

In [20]:
abstracts = [] #список для 3 клбчевых предлложений после суммаризатора LexRank

In [22]:
for text in df.text:
  doc = text
  parser = PlaintextParser.from_string(doc, Tokenizer('russian'))
  summary = summarizer(parser.document, 3)
  abstracts.append([str(sentence) for sentence in summary])

In [23]:
abstracts[:10] #объединяем предложения

[['20,77% полагают, что развитие системы дистанционного обучения будет способствовать их профессиональному росту; 27,54% не считают, что система дистанционного обучения будет способствовать их профессиональному росту [1, C.177].',
  'Но в процессе реальной коммуникации, в том числе и осуществляемой в ходе образовательной деятельности, четкого разделения в знаниях на процедурную и декларативную составляющие не существует.',
  '«Взаимодействие в процессе обучения» подразумевает тот простой факт, что люди, осуществляя познавательную деятельность, вступают в разнообразные отношения между собой и с окружающей средой.'],
 ['Существует угрозы, которые не имеют прямого отношения к социальной сети, но могут иметь серьезные последствия, как для компьютера пользователя, так и для профиля пользователя в социальной сети.',
  'Пользователь целиком и полностью определяет доступ к своим данным для любого пользователя распределенной социальной сети.',
  'В обычных социальных сетях, сервер используется 

In [24]:
annotations = []
for abstr in enumerate(abstracts, start=1):
    print(f'Аннотация статьи {abstr[0]}:')
    annotation = str(' '.join(abstr[1]))
    annotations.append(annotation)
    print(annotation, '\n\n')

Аннотация статьи 1:
20,77% полагают, что развитие системы дистанционного обучения будет способствовать их профессиональному росту; 27,54% не считают, что система дистанционного обучения будет способствовать их профессиональному росту [1, C.177]. Но в процессе реальной коммуникации, в том числе и осуществляемой в ходе образовательной деятельности, четкого разделения в знаниях на процедурную и декларативную составляющие не существует. «Взаимодействие в процессе обучения» подразумевает тот простой факт, что люди, осуществляя познавательную деятельность, вступают в разнообразные отношения между собой и с окружающей средой. 


Аннотация статьи 2:
Существует угрозы, которые не имеют прямого отношения к социальной сети, но могут иметь серьезные последствия, как для компьютера пользователя, так и для профиля пользователя в социальной сети. Пользователь целиком и полностью определяет доступ к своим данным для любого пользователя распределенной социальной сети. В обычных социальных сетях, сервер

In [25]:
annots = pd.DataFrame(annotations, columns=['annotations_sumy'])
sumy = pd.concat([df, annots], join='outer', axis=1)
sumy

,name,text,year,annotations_sumy
0,Kuzmina_IMS_2014_rus.txt,Социально-гуманитарная образовательная среда\n...,2014,"20,77% полагают, что развитие системы дистанци..."
1,Bogoraz_IMS_2014_rus.txt,Централизованные и распределенные социальные с...,2014,"Существует угрозы, которые не имеют прямого от..."
2,Grigoryeva_IMS_2014_rus.txt,Обучение пожилых ИКТ и возможности их\nтрудоус...,2014,"В то же время, исследование ФОМ пказывает, что..."
3,Dyakova_IMS_2014_rus.txt,Использование информационных технологий\nдля б...,2014,"Существует убеждение, что информационные техно..."
4,Lizunova_IMS_2014_rus.txt,Безграничность медиапространства:\nбудущее или...,2014,Медийное пространство – это пространство отнош...
...,...,...,...,...
137,Korablinov_IMS_2020_rus.txt,Подготовка набора данных для вопросно-ответног...,2020,Подготовка набора данных для вопросно-ответног...
138,Khokhlova_IMS_2020_rus.txt,Методы машинного обучения применительно к зада...,2020,Обзор методов Традиционные методы извлечения л...
139,Kuznetsova_IMS_2020_rus.txt,О возможности использования корпуса NOW в курс...,2020,Для проверки частотности из off-list списка бы...
140,Sokolova_IMS_2020_rus.txt,К вопросу о формировании набора отношений для ...,2020,Обобщение типов похожих риторических отношений...


T5

RuT5-base-sum

In [31]:
from transformers import AutoTokenizer, T5ForConditionalGeneration
import torch
from tqdm.notebook import tqdm

model_name = "IlyaGusev/rut5_base_sum_gazeta"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

annotations_t5 = []

for idx, text in enumerate(tqdm(df.text[:30], desc="Генерация аннотаций первых 30 текстов"), start=1):
    input_ids = tokenizer([text], max_length=600, add_special_tokens=True,
                          padding="max_length", truncation=True, return_tensors="pt")["input_ids"]
    output_ids = model.generate(input_ids=input_ids,
                                min_length=50,
                                max_length=250,
                                no_repeat_ngram_size=4)[0]
    summary = tokenizer.decode(output_ids, skip_special_tokens=True)
    print(f'Аннотация текста {idx}:\n{summary}\n')
    annotations_t5.append(summary)

# Для остальных текстов ставим пустые строки
annotations_t5.extend([''] * (len(df) - 30))


Генерация аннотаций первых 30 текстов:   0%|          | 0/30 [00:00<?, ?it/s]

Аннотация текста 1:
Современные информационные технологии в современной образовательной среде становятся все более востребованными и востребованными в образовательном процессе. Проблемы, связанные с разработкой и применением в учебном процессе современных компьютерных технологий, в свою очередь, обсуждаются в инженерно-техническом и педагогическом сообществах.

Аннотация текста 2:
Социальные сети стали одним из наиболее быстро развивающихся интернет-сервисов в мире. Это связано с повсеместным развитием коммуникаций, компьютеризацией общества и неистребимым желанием человека общаться со старыми друзьями, так и с новыми людьми.

Аннотация текста 3:
Пожилые люди могут оказаться в положении «потерявших роль» и экономическую независимость, считают российские ученые. Это свидетельствует о том, что старение является социально конструируемыми отношениями, практикой и процессом.

Аннотация текста 4:
Использование информационных технологий для борьбы с коррупцией и локальная административная тра

In [32]:
df['annotations_t5'] = annotations_t5

df.head(35)


,name,text,year,annotations_t5
0,Kuzmina_IMS_2014_rus.txt,Социально-гуманитарная образовательная среда\n...,2014,Современные информационные технологии в соврем...
1,Bogoraz_IMS_2014_rus.txt,Централизованные и распределенные социальные с...,2014,Социальные сети стали одним из наиболее быстро...
2,Grigoryeva_IMS_2014_rus.txt,Обучение пожилых ИКТ и возможности их\nтрудоус...,2014,Пожилые люди могут оказаться в положении «поте...
3,Dyakova_IMS_2014_rus.txt,Использование информационных технологий\nдля б...,2014,Использование информационных технологий для бо...
4,Lizunova_IMS_2014_rus.txt,Безграничность медиапространства:\nбудущее или...,2014,В настоящее время исследователи пытаются выясн...
5,Adaskina_IMS_2014_rus.txt,Полуавтоматическое пополнение словарей на осно...,2014,Метод полуавтоматического пополнения семантиче...
6,Grokhovskiy_IMS_2014_rus.txt,Лексическая база корпуса тибетских грамматичес...,2014,В Санкт-Петербургском государственном универси...
7,Zalesskaya_IMS_2014_rus.txt,Программа выявления в тексте двучленных\nстати...,2014,Программа выявления в тексте двусловных статис...
8,Kalinichenko_IMS_2014_rus.txt,Информационные технологии в целях обеспечения\...,2014,Интегрированная система информационноаналитиче...
9,Brodovskaya_Siniakov_IMS_2014_rus.txt,Влияние Интернет-коммуникации на формирование\...,2014,Российская молодая молодежь осознает политичес...


In [35]:
from tqdm.notebook import tqdm

annotations_t5_short = []

for text in tqdm(df.text[:30], desc="Генерация коротких заголовков T5 (30 шт)"):
    input_ids = tokenizer([text], max_length=512, add_special_tokens=True,
                          padding="max_length", truncation=True, return_tensors="pt")["input_ids"]
    output_ids = model.generate(
        input_ids=input_ids,
        min_length=5,
        max_length=30,
        num_beams=5,
        early_stopping=True
    )[0]
    summary = tokenizer.decode(output_ids, skip_special_tokens=True)
    annotations_t5_short.append(summary)

# Заполним остальные строки пустыми
annotations_t5_short.extend([''] * (len(df) - 30))

df['T5_short'] = annotations_t5_short

df.head(35)


Генерация коротких заголовков T5 (30 шт):   0%|          | 0/30 [00:00<?, ?it/s]

,name,text,year,annotations_t5,T5_short
0,Kuzmina_IMS_2014_rus.txt,Социально-гуманитарная образовательная среда\n...,2014,Современные информационные технологии в соврем...,Современные информационные технологии в соврем...
1,Bogoraz_IMS_2014_rus.txt,Централизованные и распределенные социальные с...,2014,Социальные сети стали одним из наиболее быстро...,Социальные сети стали одним из наиболее быстро...
2,Grigoryeva_IMS_2014_rus.txt,Обучение пожилых ИКТ и возможности их\nтрудоус...,2014,Пожилые люди могут оказаться в положении «поте...,"Ученые считают, что старение является социальн..."
3,Dyakova_IMS_2014_rus.txt,Использование информационных технологий\nдля б...,2014,Использование информационных технологий для бо...,Использование информационных технологий для бо...
4,Lizunova_IMS_2014_rus.txt,Безграничность медиапространства:\nбудущее или...,2014,В настоящее время исследователи пытаются выясн...,"Ученые и исследователи пытаются выяснить, буде..."
5,Adaskina_IMS_2014_rus.txt,Полуавтоматическое пополнение словарей на осно...,2014,Метод полуавтоматического пополнения семантиче...,Метод полуавтоматического пополнения семантиче...
6,Grokhovskiy_IMS_2014_rus.txt,Лексическая база корпуса тибетских грамматичес...,2014,В Санкт-Петербургском государственном универси...,В Санкт-Петербурге создается корпус тибетских ...
7,Zalesskaya_IMS_2014_rus.txt,Программа выявления в тексте двучленных\nстати...,2014,Программа выявления в тексте двусловных статис...,Программа выявления в тексте двусловных статис...
8,Kalinichenko_IMS_2014_rus.txt,Информационные технологии в целях обеспечения\...,2014,Интегрированная система информационноаналитиче...,Информационные технологии в целях обеспечения ...
9,Brodovskaya_Siniakov_IMS_2014_rus.txt,Влияние Интернет-коммуникации на формирование\...,2014,Российская молодая молодежь осознает политичес...,"Российская молодая молодежь, ориентированная н..."
